In [1]:
%load_ext autoreload
%autoreload 2

import polars as pl
from datasets import load_dataset
from huggingface_hub import hf_hub_download

import util.cleaning as cu

In [2]:
REPO = "phreshphish/phreshphish"

def read_hashes(filename: str) -> list[str]:
    path = hf_hub_download(REPO, filename, repo_type="dataset")
    with open(path) as f:
        return f.read().split()

def make_combined(train: pl.DataFrame, benchmark: pl.DataFrame) -> pl.DataFrame:
    return pl.concat([
        train.with_columns(split=pl.lit("train")),
        benchmark.with_columns(split=pl.lit("benchmark")),
    ], how="vertical_relaxed")

# the full benchmark is every test sample listed in benign.txt or phish.txt
benchmark_hashes = set(read_hashes("benchmark/benign.txt") + read_hashes("benchmark/phish.txt"))

train = load_dataset(REPO, split="train").to_polars()
benchmark = (
    load_dataset(REPO, split="test")
    .filter(lambda shas: [s in benchmark_hashes for s in shas], input_columns="sha256", batched=True)
    .to_polars()
)

Resolving data files:   0%|          | 0/56 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/56 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/96 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/56 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/56 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/27 [00:00<?, ?it/s]

In [3]:
df = make_combined(train, benchmark)

# the data has no title column, so parse it from the HTML
df = cu.add_titles(df, n_jobs=4, batch_size=256)

Extracting titles: 100%|██████████| 616993/616993 [07:40<00:00, 1339.61rows/s]


In [4]:
print("Appling automated heuristics...")
df = cu.find_duplicate_urls(df)
df = cu.find_empty_html(df)
df = cu.find_bad_titles(df)

Appling automated heuristics...
Found 0 subsequent duplicate URLs
Found 0 empty HTML documents
Found 0 bad titles


In [5]:
df = df.filter(~(pl.col('bad_title')) & (~pl.col('empty_html'))).drop(['empty_html', 'bad_title'])
df = df.with_row_index()
df.group_by('label').agg(pl.col('label').count().alias('n'))

label,n
str,u32
"""phish""",250002
"""benign""",366991


In [ ]:
X_tfidf = cu.build_tfidf(df, max_features=None, sample_frac=0.01, vectorizer_path="./tfidf_vectorizer.joblib")

In [ ]:
lsh_model = cu.train_lsh(X_tfidf, n_vectors=32)

In [ ]:
# add the bin_id to dataframe
mapping = pl.DataFrame(
    [
        {"index": idx, "bin": bin_id}
        for bin_id, indices in lsh_model["table"].items()
        for idx in indices
    ]
)

df = df.join(mapping, on="index", how="left")

In [ ]:
cleaned_1 = cu.run_cleaning(X_tfidf, df, grouping_method="bin", lsh_model=lsh_model, budget=30)

In [ ]:
# run a second time to go back to largest bins
cleaned_2 = cu.run_cleaning(X_tfidf, cleaned_1, grouping_method="bin", lsh_model=lsh_model, budget=30)

In [ ]:
# how many of each class were removed at this stage?
reject_stats = (
    cleaned_2
    # only consider rows where a decision was made
    .filter(pl.col("keep").is_not_null())
    # group by the class label
    .group_by("label")
    .agg([
        # how many were rejected?
        pl.col("keep").eq(False).sum().alias("num_rejected"),
    ])
)
reject_stats

In [ ]:
# now, switch to "title" grouping scheme.
# we run twice with budget=50 to account for the fact
# that we may encounter the same title more than once
# and we want to ensure that we process at least 50
# unique titles
cleaned_3 = cu.run_cleaning(X_tfidf, cleaned_2, grouping_method="title", lsh_model=lsh_model, budget=50)

In [ ]:
cleaned_4 = cu.run_cleaning(X_tfidf, cleaned_3, grouping_method="title", lsh_model=lsh_model, budget=50)

In [ ]:
# how many of each class were removed at this stage?
reject_stats = (
    cleaned_4
    # only consider rows where a decision was made
    .filter(pl.col("keep").is_not_null())
    # group by the class label
    .group_by("label")
    .agg([
        # how many were rejected?
        pl.col("keep").eq(False).sum().alias("num_rejected"),
    ])
)
reject_stats